# Evidence snapshot (task YL-004B, stage 02)

Takes the address of a public company page and makes a **dated text snapshot** of it. No AI is used here: the text is copied as it is, with the date it was read and the date the page shows.
The result is a block you paste into the field `evidence` of the Make scenario "YL-004B | Teaser Generator (draft)".

**How to use:** run the cells from top to bottom. Nothing is sent anywhere except the page requests. Public pages only, https only; pages that forbid it in `robots.txt` are skipped.

### Step 1. Get the code (branch `lab-step3`)
If the folder already exists from an earlier session, the cell updates it.

In [ ]:
import os

REPO_URL = "https://github.com/Yul04ek/ai-content-claim-check.git"
BRANCH = "lab-step3"
FOLDER = "/content/claim-check"

if not os.path.exists(FOLDER):
    # A private repository needs a token. A public one works without it.
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        clone_url = REPO_URL.replace("https://", f"https://{token}@")
    except Exception:
        clone_url = REPO_URL
    !git clone -q -b {BRANCH} {clone_url} {FOLDER}
    !git -C {FOLDER} remote set-url origin {REPO_URL}   # do not keep the token in the repo settings
else:
    !git -C {FOLDER} fetch -q origin {BRANCH}
    !git -C {FOLDER} checkout -q {BRANCH}
    !git -C {FOLDER} pull -q origin {BRANCH}
%cd {FOLDER}
!git log --oneline -1
!pip install -q requests beautifulsoup4 lxml
!python tests/test_evidence_fetch.py | tail -1

### Step 2. Read the pages
Put the page addresses in the list (one per line, https only). Each address becomes one source, S1, S2 and so on, in this order.
The cell prints for every page: how many words were found, the date the page shows, and the first lines of the text.

In [ ]:
import sys
sys.path.insert(0, "/content/claim-check")
from evidence_fetch.fetch import fetch_snapshot

URLS = [
    "https://cal.com/about",
    # "https://cal.com/blog/seed",
]

snaps = []
for url in URLS:
    try:
        s = fetch_snapshot(url)
    except Exception as e:
        print(f"SKIPPED {url}\n   reason: {e}\n")
        continue
    snaps.append(s)
    words = len(s["text"].split())
    print(f"OK  {url}")
    print(f"   words: {words} | page date: {s['page_date'] or 'none shown (undated)'} | emails on the page: {s['emails_found']}")
    if words < 80:
        print("   WARNING: very little text. The page may be built by scripts. Use Step 4 (saved page) instead.")
    print("   first lines:")
    print("   " + "\n   ".join(s["text"].splitlines()[:8]))
    print()
print(f"{len(snaps)} of {len(URLS)} pages read")

### Step 3. The evidence block for Make
This prints the whole block. Copy everything between the two lines into the field `evidence`. The same block is also saved as a file (the cell offers the download).
Before pasting, look through the text once: is it the text of the page, without menus or cookie banners?

In [ ]:
import json
from pathlib import Path
from evidence_fetch.fetch import to_source_block, to_pack_source

assert snaps, "No page was read. Check Step 2."
block = "\n".join(to_source_block(s) for s in snaps)
out = Path("snapshots"); out.mkdir(exist_ok=True)
(out / "make_evidence.txt").write_text(block, encoding="utf-8")
(out / "snapshots.json").write_text(json.dumps(snaps, ensure_ascii=False, indent=1), encoding="utf-8")
(out / "pack_sources.json").write_text(
    json.dumps([to_pack_source(s, f"S{i}") for i, s in enumerate(snaps, 1)], ensure_ascii=False, indent=1), encoding="utf-8")

print("=" * 20, "COPY FROM HERE", "=" * 20)
print(block)
print("=" * 20, "TO HERE", "=" * 20)
print(f"{len(block.split())} words in total")

try:
    from google.colab import files
    files.download(str(out / "make_evidence.txt"))
except Exception:
    pass

### Step 4 (only if Step 2 gave little or no text). A saved page instead
Open the page in the browser, save it (Ctrl+S, "Webpage, HTML only") and upload the file here. The cell does the same cleaning as Step 2.
Type the original address of the page in `URL`; it goes into the source line.

In [ ]:
from google.colab import files
from evidence_fetch.fetch import snapshot_from_html, to_source_block

URL = "https://cal.com/about"   # the original address of the saved page

uploaded = files.upload()       # choose the saved .html file
name = next(iter(uploaded))
snap = snapshot_from_html(uploaded[name].decode("utf-8", errors="replace"), URL)
print(f"words: {len(snap['text'].split())} | page date: {snap['page_date'] or 'none shown (undated)'}")
print("=" * 20, "COPY FROM HERE", "=" * 20)
print(to_source_block(snap))
print("=" * 20, "TO HERE", "=" * 20)